In [6]:
import pandas as pd
import requests
import time

# --- CONFIG ---
API_KEY = 'yIgrwG6fkTDnup5te4JqKlaVC8qe9YB8'  # <-- Replace this with your actual FMP API key
BATCH_SIZE = 100  # FMP supports batch requests up to ~100 symbols per call
API_URL = 'https://financialmodelingprep.com/api/v3/profile/'

# --- Load tickers ---
df = pd.read_csv('matching.csv', dtype={'FIGI': str})
sp500_tickers = df['ticker'].unique()

# --- Helper function ---
def fetch_sector_batch(tickers, api_key):
    ticker_str = ','.join(tickers)
    url = f"{API_URL}{ticker_str}?apikey={api_key}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return {item['symbol']: item.get('sector', 'Unknown') for item in data}
        else:
            print(f"Failed batch: {tickers} — HTTP {response.status_code}")
    except Exception as e:
        print(f"Exception for batch {tickers}: {e}")
    return {ticker: 'Unknown' for ticker in tickers}

# --- Batch process ---
ticker_to_sector = {}
for i in range(0, len(sp500_tickers), BATCH_SIZE):
    batch = sp500_tickers[i:i+BATCH_SIZE]
    sectors = fetch_sector_batch(batch, API_KEY)
    ticker_to_sector.update(sectors)
    print(f"Processed batch {i//BATCH_SIZE + 1}: {len(batch)} tickers.")
    time.sleep(1)  # Avoid rapid-fire requests

# --- Map sectors to original DataFrame ---
df['sector'] = df['ticker'].map(ticker_to_sector).fillna('Unknown')

# --- Save result ---
df.to_csv('matching_with_sector.csv', index=False)
print("✅ Sector information added and file saved.")


Processed batch 1: 100 tickers.
Processed batch 2: 100 tickers.
Processed batch 3: 100 tickers.
Processed batch 4: 100 tickers.
Processed batch 5: 100 tickers.
Processed batch 6: 3 tickers.
✅ Sector information added and file saved.


In [2]:
import pandas as pd

df = pd.read_csv('sp500_final_common_cleaned.csv', dtype={'FIGI': str})
df_matching = pd.read_csv('matching_with_sector.csv', dtype={'FIGI': str})
cusip_to_sector = dict(zip(df_matching['CUSIP'].str.upper(), df_matching['sector']))

num_holders = df.groupby(['CUSIP', 'source_file'])['CIK'].nunique().reset_index(name='num_holders')
# Total institutional ownership (VALUE in $1,000s, convert to USD)
total_ownership = df.groupby(['CUSIP', 'source_file'])['VALUE'].sum().reset_index(name='total_institutional_ownership')
# Total shares held (proxy for Percent of Float Owned)
total_shares = df.groupby(['CUSIP', 'source_file'])['SSHPRNAMT'].sum().reset_index(name='total_shares_held')
# Average holding per manager (VALUE in $1,000s, convert to USD)
avg_holding = df.groupby(['CUSIP', 'source_file'])['VALUE'].mean().reset_index(name='avg_holding_per_manager')
# Volatility of holdings (standard deviation of VALUE, in $1,000s, convert to USD)
volatility = df.groupby(['CUSIP', 'source_file'])['VALUE'].std().reset_index(name='volatility_of_holdings')
# Sector
cusip_features = num_holders.copy()
cusip_features['sector'] = cusip_features['CUSIP'].map(cusip_to_sector).fillna('Unknown')

# Merge features
cusip_features = cusip_features.merge(total_ownership, on=['CUSIP', 'source_file'], how='left')
cusip_features = cusip_features.merge(total_shares, on=['CUSIP', 'source_file'], how='left')
cusip_features = cusip_features.merge(avg_holding, on=['CUSIP', 'source_file'], how='left')
cusip_features = cusip_features.merge(volatility, on=['CUSIP', 'source_file'], how='left')
# Reorder columns
cusip_features = cusip_features[['CUSIP', 'source_file', 'num_holders', 'total_institutional_ownership', 'total_shares_held', 'avg_holding_per_manager', 'volatility_of_holdings', 'sector']]

df_cusip = pd.DataFrame(cusip_features)
df_cusip = df_cusip.sort_values(by=['source_file', 'CUSIP'])

df_cusip.to_csv('cusip_features.csv', index=False)
df_cusip.head(50)

,CUSIP,source_file,num_holders,total_institutional_ownership,total_shares_held,avg_holding_per_manager,volatility_of_holdings,sector
141,002824100,2013q2_form13f.csv,1,1183,33500,1183.000000,NaN,Healthcare
189,00287Y109,2013q2_form13f.csv,1,1737,42600,1737.000000,NaN,Healthcare
1407,032654105,2013q2_form13f.csv,1,2566,55205,2566.000000,NaN,Technology
1571,037833100,2013q2_form13f.csv,1,3522,7958,3522.000000,NaN,Technology
4147,166764100,2013q2_form13f.csv,1,2605,21923,2605.000000,NaN,Energy
4564,191216100,2013q2_form13f.csv,1,3502,86596,3502.000000,NaN,Consumer Defensive
5562,244199105,2013q2_form13f.csv,1,1303,15150,1303.000000,NaN,Industrials
6488,278865100,2013q2_form13f.csv,1,2017,25157,2017.000000,NaN,Basic Materials
7314,30231G102,2013q2_form13f.csv,1,3426,38021,3426.000000,NaN,Energy
7786,337738108,2013q2_form13f.csv,1,1245,14175,1245.000000,NaN,Technology


In [ ]:
import pandas as pd

df = pd.read_csv('cusip_features.csv')
df = df.fillna(0)
df.info()
print(df.isna().sum())
print(df.duplicated().sum())

df = df.sort_values(by=['source_file', 'CUSIP'])

df.to_csv('cusip_features.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21009 entries, 0 to 21008
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   CUSIP                          21009 non-null  object 
 1   source_file                    21009 non-null  object 
 2   num_holders                    21009 non-null  int64  
 3   total_institutional_ownership  21009 non-null  int64  
 4   total_shares_held              21009 non-null  int64  
 5   avg_holding_per_manager        21009 non-null  float64
 6   volatility_of_holdings         21009 non-null  float64
 7   sector                         21009 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 1.3+ MB
CUSIP                            0
source_file                      0
num_holders                      0
total_institutional_ownership    0
total_shares_held                0
avg_holding_per_manager          0
volatility_of_holdin

In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('cusip_features.csv')
encoder = LabelEncoder()
df['sector'] = encoder.fit_transform(df['sector'])
df.info()
df = df.sort_values(by=['source_file', 'CUSIP'])
df.to_csv('cusip_features.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21009 entries, 0 to 21008
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   CUSIP                          21009 non-null  object 
 1   source_file                    21009 non-null  object 
 2   num_holders                    21009 non-null  int64  
 3   total_institutional_ownership  21009 non-null  int64  
 4   total_shares_held              21009 non-null  int64  
 5   avg_holding_per_manager        21009 non-null  float64
 6   volatility_of_holdings         21009 non-null  float64
 7   sector                         21009 non-null  int32  
dtypes: float64(2), int32(1), int64(3), object(2)
memory usage: 1.2+ MB


In [11]:
import pandas as pd

df = pd.read_csv('cusip_features.csv')
df.info()
print(df.isna().sum())
print(df.duplicated(subset=['CUSIP', 'source_file']).sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21009 entries, 0 to 21008
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   CUSIP                          21009 non-null  object 
 1   source_file                    21009 non-null  object 
 2   num_holders                    21009 non-null  int64  
 3   total_institutional_ownership  21009 non-null  int64  
 4   total_shares_held              21009 non-null  int64  
 5   avg_holding_per_manager        21009 non-null  float64
 6   volatility_of_holdings         21009 non-null  float64
 7   sector                         21009 non-null  int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 1.3+ MB
CUSIP                            0
source_file                      0
num_holders                      0
total_institutional_ownership    0
total_shares_held                0
avg_holding_per_manager          0
volatility_of_holdin

In [ ]:
import pandas as pd

df = pd.read_csv('cusip_features.csv')
print(df['sector'].unique())
df = df.assign(sector = df['sector'] + 1)
print(df['sector'].unique())
df.head()
#df.to_csv('cusip_features.csv', index=False)

[ 6  9  4  3  7  0  2  5 10  1  8]
[ 7 10  5  4  8  1  3  6 11  2  9]


In [44]:
import pandas as pd

df_cusip = pd.read_csv('cusip_features.csv')
df_fixed = df_cusip[df_cusip['source_file'] == '2025q1_form13f.csv'].copy()
set1 = df_fixed['CUSIP'].unique()
df_ref = pd.read_csv('matching_with_sector.csv')
set2 = df_ref['CUSIP'].unique()
set3 = set(set2) - set(set1)
print(f"Number of CUSIPs in reference: {len(set2)}")
print(f"Number of CUSIPs in fixed: {len(set1)}")
print(f"Number of CUSIPs in set3: {len(set3)}")
print(f"Set3: {set3}")
df_con = df_cusip[df_cusip['CUSIP'].isin(set3)]
df_con = df_con[df_con['source_file'] == '2023q4_form13f.csv']
#print(df_con)
df_fixed2 = pd.concat([df_fixed, df_con], ignore_index=True)
df_fixed2 = df_fixed2[['CUSIP', 'sector']]
#df_fixed2.info()
#print(len(df_fixed2['CUSIP'].unique()))
df_fixed2.to_csv('fixed_values.csv', index=False)

Number of CUSIPs in reference: 503
Number of CUSIPs in fixed: 501
Number of CUSIPs in set3: 2
Set3: {'143658300', 'G5494J103'}


In [45]:
import pandas as pd

df_cusip = pd.read_csv('cusip_features.csv')
df_cusip.head()

,CUSIP,source_file,num_holders,total_institutional_ownership,total_shares_held,avg_holding_per_manager,volatility_of_holdings,sector
0,002824100,2013q2_form13f.csv,1,1183,33500,1183.0,0.0,7
1,00287Y109,2013q2_form13f.csv,1,1737,42600,1737.0,0.0,7
2,032654105,2013q2_form13f.csv,1,2566,55205,2566.0,0.0,10
3,037833100,2013q2_form13f.csv,1,3522,7958,3522.0,0.0,10
4,166764100,2013q2_form13f.csv,1,2605,21923,2605.0,0.0,5


In [ ]:
import pandas as pd
from tgb.nodeproppred.dataset import NodePropPredDataset

dataset = NodePropPredDataset(name='tgbn-13f', root='datasets')
df = pd.read_csv('cusip_features.csv')
dict_cik = dict(zip(df['source_file'].unique(), dataset.label_ts))
df['source_file'] = df['source_file'].map(dict_cik)
df.to_csv('cusip_features.csv', index=False)
df.head(50)

raw file found, skipping download
Dataset directory is  c:\Users\emadi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tgb/datasets\tgbn_13f
loading processed file


,CUSIP,source_file,num_holders,total_institutional_ownership,total_shares_held,avg_holding_per_manager,volatility_of_holdings,sector
0,002824100,20132,1,1183,33500,1183.000000,0.000000,7
1,00287Y109,20132,1,1737,42600,1737.000000,0.000000,7
2,032654105,20132,1,2566,55205,2566.000000,0.000000,10
3,037833100,20132,1,3522,7958,3522.000000,0.000000,10
4,166764100,20132,1,2605,21923,2605.000000,0.000000,5
5,191216100,20132,1,3502,86596,3502.000000,0.000000,4
6,244199105,20132,1,1303,15150,1303.000000,0.000000,8
7,278865100,20132,1,2017,25157,2017.000000,0.000000,1
8,30231G102,20132,1,3426,38021,3426.000000,0.000000,5
9,337738108,20132,1,1245,14175,1245.000000,0.000000,10


In [48]:
import pandas as pd
from tgb.nodeproppred.dataset import NodePropPredDataset

dataset = NodePropPredDataset(name='tgbn-13f', root='datasets')
df = pd.read_csv('cusip_features.csv')
df.info()
print(df_cusip[(df_cusip['CUSIP'] == '001055102') &
                              (df_cusip['source_file'] == 20132)].iloc[:, 2:].values)

raw file found, skipping download
Dataset directory is  c:\Users\emadi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tgb/datasets\tgbn_13f
loading processed file
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21009 entries, 0 to 21008
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   CUSIP                          21009 non-null  object 
 1   source_file                    21009 non-null  int64  
 2   num_holders                    21009 non-null  int64  
 3   total_institutional_ownership  21009 non-null  int64  
 4   total_shares_held              21009 non-null  int64  
 5   avg_holding_per_manager        21009 non-null  float64
 6   volatility_of_holdings         21009 non-null  float64
 7   sector                         21009 non-null  int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 1.3+ MB
[]
